# 01 - Pembersihan Data & Persiapan Dataset

**Alur Proses:**

1. Memecah teks panjang dari transkripsi AI menjadi segmen-segmen kalimat (Chunking).
2. Pembersihan teks dari karakter rusak dan perbaikan ejaan.
3. Pemisahan data (Latih & Uji) berbasiskan Dokumen untuk memastikan independensi data.


## 1. Import Library

In [2]:
import pandas as pd
import re
from pathlib import Path
import sys
sys.path.append('..')
from modules.ocr_risalah import proses_semua_pdf

## Proses OCR Dokumen (Referensi) (optional)
Bagian ini hanya memindai dokumen asli PDF untuk kebutuhan referensi validasi ejaan. Kode utama tidak bergantung pada tahapan ini.

In [2]:
DIR_PDF    = Path('../dataset/01_raw/risalah_pdf')
DIR_OCR    = Path('../dataset/02_extracted/ocr_risalah')

# Menentukan batas awal dan akhir dari paragraf yang diambil
POLA_AWAL  = r'MENYANYIKAN LAGU INDONESIA RAYA'
POLA_AKHIR = r'RAPAT DITUTUP PUKUL'

hasil_ocr = proses_semua_pdf(
    direktori_pdf=DIR_PDF,
    direktori_output=DIR_OCR,
    pola_awal=POLA_AWAL,
    pola_akhir=POLA_AKHIR,
)
print(f'Total risalah berhasil di-OCR: {len(hasil_ocr)}')

✅ Paripurna_Ke_10_Persidangan_II_2024_2025.pdf → 4094 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_II_2024_2025.pdf → 711 kata diekstrak
✅ Paripurna_Ke_11_Persidangan_III_2023_2024.pdf → 2469 kata diekstrak
✅ Paripurna_Ke_12_Persidangan_III_2023_2024.pdf → 3676 kata diekstrak
✅ Paripurna_Ke_13_Persidangan_IV_2023_2024.pdf → 6099 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_II_2024_2025.pdf → 1613 kata diekstrak
✅ Paripurna_Ke_14_Persidangan_IV_2023_2024.pdf → 9899 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_II_2024_2025.pdf → 6006 kata diekstrak
✅ Paripurna_Ke_15_Persidangan_IV_2023_2024.pdf → 2325 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_II_2024_2025.pdf → 2543 kata diekstrak
✅ Paripurna_Ke_16_Persidangan_V_2023_2024.pdf → 3956 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_III_2024_2025.pdf → 852 kata diekstrak
✅ Paripurna_Ke_17_Persidangan_V_2023_2024.pdf → 5023 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_III_2024_2025.pdf → 5208 kata diekstrak
✅ Paripurna_Ke_18_Persidangan_V_20

## 2. Pemotongan Teks (Chunking)
Teks transkripsi lengkap dari Whisper dipotong per kalimat hingga batas panjang maksimal. Untuk Mengatasi OOM pada model IndoT5 nantinya

In [ ]:
import pandas as pd
import re
from pathlib import Path

DIR_TRANSKRIP = Path('../dataset/02_extracted/whisper_transcripts')
DIR_OCR = Path('../dataset/02_extracted/ocr_risalah')
MAX_WORDS = 350 

baris_data = []

print("Sedang melakukan proses chunking...")

for txt_file in sorted(DIR_TRANSKRIP.glob('*.txt')):
    source_whisper = txt_file.read_text(encoding='utf-8').strip()
    
    # Mengecek ketersediaan file referensi terkait (jika ada)
    ocr_file = DIR_OCR / (txt_file.stem + '.txt')
    
    if not source_whisper:
        continue
        
    if not ocr_file.exists():
        print(f"OCR tidak ditemukan untuk {txt_file.stem}, dilewati.")
        continue

    kalimat_list = re.split(r'(?<=[.!?]) +', source_whisper)
    chunk_saat_ini = []
    jumlah_kata_saat_ini = 0

    for kalimat in kalimat_list:
        jml_kata = len(kalimat.split())

        if jumlah_kata_saat_ini + jml_kata > MAX_WORDS and jumlah_kata_saat_ini > 0:
            baris_data.append({
                "dokumen_asal": txt_file.stem,
                "input_whisper_segment": " ".join(chunk_saat_ini),
                "target_summary_manual": ""
            })
            chunk_saat_ini = [kalimat]
            jumlah_kata_saat_ini = jml_kata
        else:
            chunk_saat_ini.append(kalimat)
            jumlah_kata_saat_ini += jml_kata

    if chunk_saat_ini and jumlah_kata_saat_ini > 50:
        baris_data.append({
            "dokumen_asal": txt_file.stem,
            "input_whisper_segment": " ".join(chunk_saat_ini),
            "target_summary_manual": ""
        })

df = pd.DataFrame(baris_data)
df.to_csv('../dataset/data_segment_siap_anotasi.csv', index=False)

print(f"Selesai! {len(df)} segmen siap dianotasi.")

Sedang memotong Whisper dan memvalidasi keberadaan OCR...
Selesai! 486 segmen siap dianotasi.


## 3. Pembersihan Teks (Cleaning)
Memulihkan teks dengan membuang simbol aneh akibat kesalahan encoding, dan mengoreksi kata (typo) secara otomatis.

In [2]:
import pandas as pd
import re

df = pd.read_csv('../dataset/data_segment_siap_anotasi.csv')

def clean_whisper(text):
    text = str(text).lower()

    # Hapus karakter encoding rusak di awal (misal: sÃ¶ylems esta)
    text = re.sub(r'^[^\w\s]+', '', text)

    # Membersihkan karakter simbol yang tidak standar
    text = re.sub(r'[^\x00-\x7F\u00C0-\u024F\s]', ' ', text)

    # Menyingkirkan bentuk teks yang bukan huruf atau angka
    text = re.sub(r'[^\w\s.,!?-]', ' ', text)

    # Merapatkan jarak spasi berlebih pada pungtuasi
    text = re.sub(r'\s+([.,!?])', r'\1', text)

    # Gabungkan angka terpisah (contoh: 154 . 1 → 154,1)
    text = re.sub(r'(\d)\s*\.\s*(\d)', r'\1,\2', text)

    # Hapus pengulangan kata (contoh: yang yang → yang)
    text = re.sub(r'\b(\w+)\s+\1\b', r'\1', text)

    # Merapikan spasi ganda menjadi tunggal
    text = re.sub(r'\s+', ' ', text).strip()

    return text

df['input_whisper_segment'] = df['input_whisper_segment'].apply(clean_whisper)

df.to_csv('../dataset/data_segment_cleaned.csv', index=False)
print("Cleaning selesai. Dataset cleaned tersimpan.")

Cleaning selesai. Dataset cleaned tersimpan.


In [ ]:
import pandas as pd
import re
from pathlib import Path
from rapidfuzz import process, fuzz

DIR_OCR = Path('../dataset/02_extracted/ocr_risalah')

df = pd.read_csv('../dataset/data_segment_cleaned.csv')

def build_ocr_vocab(dokumen):
    ocr_file = DIR_OCR / f"{dokumen}.txt"
    if not ocr_file.exists():
        return set()
    text = ocr_file.read_text(encoding="utf-8").lower()
    words = re.findall(r'\b\w+\b', text)
    return set(words)

def correct_segment(row):
    vocab = build_ocr_vocab(row["dokumen_asal"])
    if not vocab:
        return row["input_whisper_segment"]

    words = row["input_whisper_segment"].split()
    corrected_words = []

    for w in words:
        if w in vocab:
            # Kata lolos validasi
            corrected_words.append(w)
        elif len(w) <= 3:
            # Kata pendek (≤3 huruf) jangan dikoreksi, terlalu berisiko salah
            corrected_words.append(w)
        else:
            # Mencari kecocokan terdekat (string-matching) berdasarkan rasio
            # Contoh: "rami" → "kami" (75), "siddang" → "sidang" (92)
            match = process.extractOne(
                w,
                vocab,
                scorer=fuzz.ratio  # lebih akurat untuk kata tunggal vs token_sort_ratio
            )
            if (match
                    and match[1] >= 75                          # Batas ambang kemiripan yang diizinkan untuk dikoreksi
                    and abs(len(w) - len(match[0])) <= 2):     # Selisih rasio panjang kata
                corrected_words.append(match[0])
            else:
                corrected_words.append(w)

    return " ".join(corrected_words)

df["input_whisper_segment"] = df.apply(correct_segment, axis=1)

df.to_csv('../dataset/data_segment_corrected_ocr.csv', index=False)
print("proses correction selesai.")

OCR-guided correction selesai.


## 4. Pemisahan Dataset (Train-Test Split)
Memilah dataset menjadi porsi pembelajaran model (80%) dan evaluasi (20%).

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load data yang sudah di proses chunk di atas
df = pd.read_csv('../dataset/data_segment_corrected_ocr.csv')

# Membuang deretan dataset bernilai kosong
df = df.dropna(subset=['input_whisper_segment', 'target_summary_manual'])
df = df[df['target_summary_manual'].str.strip() != ""].reset_index(drop=True)

print(f'Data siap latih: {len(df)} baris')
print(f'Jumlah dokumen unik: {df["dokumen_asal"].nunique()}')

# Split berdasarkan DOKUMEN, bukan baris
# Bertujuan memastikan agar tidak ada satu dokumen rapat yang tumpang tindih
dokumen_unik = df['dokumen_asal'].unique()

train_docs, test_docs = train_test_split(
    dokumen_unik,
    test_size=0.20,
    random_state=42,
    shuffle=True,
)

df_train = df[df['dokumen_asal'].isin(train_docs)].reset_index(drop=True)
df_test  = df[df['dokumen_asal'].isin(test_docs)].reset_index(drop=True)

print(f'\nDokumen train : {len(train_docs)} dokumen')
print(f'Dokumen test  : {len(test_docs)} dokumen')
print(f'\nBaris train   : {len(df_train)} ({len(df_train)/len(df)*100:.1f}%)')
print(f'Baris test    : {len(df_test)} ({len(df_test)/len(df)*100:.1f}%)')

# Memvalidasi ulang jika terdapat kebocoran dokumen
assert len(set(train_docs) & set(test_docs)) == 0, "Ada overlap dokumen!"
print('\n✅ Tidak ada overlap dokumen antara train dan test')

Data siap latih: 486 baris
Jumlah dokumen unik: 30

Dokumen train : 24 dokumen
Dokumen test  : 6 dokumen

Baris train   : 390 (80.2%)
Baris test    : 96 (19.8%)

✅ Tidak ada overlap dokumen antara train dan test


## 5. Render File Dataset Akhir
Mengekspor seluruh file hasil preparasi.

In [3]:
from pathlib import Path

DATA_DIR = Path('../dataset/03_paired')
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Mengekspor kolom inti yang digunakan di skema pelatihan
kolom = ['dokumen_asal','input_whisper_segment', 'target_summary_manual']

df_train[kolom].to_csv(DATA_DIR / 'train.csv', index=False, encoding='utf-8')
df_test[kolom].to_csv(DATA_DIR  / 'test.csv',  index=False, encoding='utf-8')

print('Dataset berhasil disimpan:')
print(f'  train.csv → {len(df_train)} baris')
print(f'  test.csv  → {len(df_test)} baris')
print(f'  Lokasi    : {DATA_DIR.resolve()}')


Dataset berhasil disimpan:
  train.csv → 390 baris
  test.csv  → 96 baris
  Lokasi    : D:\Skripsi\meeting-summarizer\dataset\03_paired
